# Gradiente Descencente para polinomio de grau 17
O presente código demonstra a implementação de um algoritmo com o intuito de obter os coeficientes de uma função polinomial de grau $n$.

O código será aplicado em uma sequência de polinomios para verificação de acurácia.

## Imports
Utilizaremos tensorflow para obter as derivadas parciais da loss function `f_loss` e para operações com os coeficientes `a`.

In [2]:
import tensorflow as tf
import random

2024-11-01 18:31:28.021583: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Funções

### `gerar_sequencia`
`gerar_sequencia` é uma função de apoio para criação do dataset de entrada (para $x$). Obtem uma sequência de `num_elementos` números igualmente distribuidos entre `valor_inicial` e `valor_final`.

In [3]:
def gerar_sequencia(num_elementos, valor_inicial, valor_final):
    return [valor_inicial + (valor_final - valor_inicial) * i / (num_elementos - 1) for i in range(num_elementos)]

### Função `f`
A função `f` representa a função polinomial f(x) de grau $a-1$ (sendo `a` o número de coeficientes)

#### Exemplo
Se o coeficientes `a` forem apenas dois elementos, teremos:

`range` terá `j` como (`0`,`1`) (dois elementos) e `n_a` terá valor igual a `2`.

para $a[j]\cdot x^{(n_a-1)-j}$

sendo `j = 0`, $\qquad a[0]\cdot x^{(2-1)-0}=a[0]\cdot x^1$

sendo `j = 1`, $\qquad a[1]\cdot x^{(2-1)-1}=a[1]\cdot x^0$

Ou Seja
`f` retornará a somatória: 
$$
f(x) = a[0]\cdot x^1+a[1]\cdot x^0 = a_1x+a_2
$$
(função da reta)
#### Utilização
Utilizamos a função `f` tanto para a função de erro `f_loss` quanto para geração de elementos do dataset de entrada `dataset_y`.

In [11]:
def f(a,x):
    n_a = a.shape[0]
    return tf.reduce_sum(a[j] * (x ** ((n_a-1) - j)) for j in range(n_a))

### função de perda (erro quadrático) `f_loss`
Utilizamos `f_loss` para minimizar o erro para encontrar os melhores parâmetros do modelo
#### Exemplo
`f_loss` para polinômio de grau 17.
$$\begin{align}
&(y_1-(a_1x_1^{17}+a_2x_1^{16}+a_3x_1^{15}+a_4x_1^{14}+a_5x_1^{13}+a_6x_1^{12}+a_7x_1^{11}\\
&+a_8x_1^{10}+a_9x_1^{9}+a_{10}x_1^{8}+a_{11}x_1^{7}+a_{12}x_1^{6}+a_{13}x_1^{5}+a_{14}x_1^{4}\\
&+a_{15}x_1^{3}+a_{16}x_1^{2}+a_{17}x_1+a_{18}))^2\\
&+\cdots+\\
&(y_n-(a_1x_n^{17}+a_2x_n^{16}+a_3x_n^{15}+a_4x_n^{14}+a_5x_n^{13}+a_6x_n^{12}+a_7x_n^{11}+a_8x_n^{10}\\
&+a_9x_n^{9}+a_{10}x_n^{8}+a_{11}x_n^{7}+a_{12}x_n^{6}+a_{13}x_n^{5}+a_{14}x_n^{4}+a_{15}x_n^{3}\\
&+a_{16}x_n^{2}+a_{17}xn+a_{18}))^2
\end{align}$$

In [5]:
def f_loss(a, x, y):
    n_a = a.shape[0]
    l = 0.0
    for i in range(y.shape[0]):
        polinomial = f(a,x[i])
        l += (y[i] - polinomial) ** 2
    return l

### Função Gradientes `grad`
A função `grad` calcula os gradientes da função `f_loss` em relação aos coeficientes `a` do polinômio usando o TensorFlow.

In [6]:
def grad(a,x,y):
    with tf.GradientTape() as tape:
        loss = f_loss(a, x, y)

    grads = tape.gradient(loss, a)
    return grads

### Distância entre coeficientes `dist`
A função `dist` calcula a distância entre os coeficientes finais e iniciais.

In [7]:
def dist(a_n, a_o):
    return tf.norm(a_n - a_o).numpy()

### Gradiente Descendente `grad_desc`
A função `grad_desc` implementa o algoritmo de gradiente descendente para otimizar os coeficientes de um polinômio de grau especificado, utilizando dados de entrada `dataset_x` e `dataset_y`. 

Utilizamos a abordagem de coeficientes iniciais aleatórios para facilitar no caso de polinômios grandes.

A função calcula os gradientes da função de perda com respeito aos coeficiente, e os atualiza com uma taxa de aprendizado `lr`, até que a distância entre os coeficientes gerados e os coeficientes anteriores seja menor que um valor de tolerância `tol`. 

A função retorna os coeficientes otimizados e o número total de iterações realizadas.

In [57]:
def grad_desc(grau, dataset_x, dataset_y,tol,lr):
    coefs=grau+1
    x = tf.constant(dataset_x, dtype=tf.float32)  # Example input values
    y = tf.constant(dataset_y, dtype=tf.float32)
    
    # coeficientes iniciais aleatórios
    a_o =  [round(random.uniform(-5, 5), 2) for _ in range(coefs)]
    a_o = tf.Variable(a_o, dtype=tf.float32, name='coefs_init')

    a_n = tf.Variable([0.0 for _ in range(coefs)]) # iniciais
    i = 0
    
    while True:
        gradients = grad(a_n, x, y)  # Get gradients for all coefficients
        a_n.assign(a_n - lr * gradients)  # Update all coefficients
        
        i += 1
        if dist(a_n, a_o) > tol:
            a_o.assign(a_n)  # Update the old coefficients to the new ones
        else:
            break
    return a_o.numpy(), i

### Cálculo de Acurácia
A função `calcular_acuracia` recebe os dados de entrada `dataset_x`, `dataset_y` e os coeficientes `coefs` gerados pela função `grad_desc`.

`calcular_acuracia` gera uma lista de previsões de y por `dataset_x`, calcula o erro absoluto médio de `dataset_y` e `previsoes` e o retorna.

In [27]:
def calcular_acuracia(dataset_x, dataset_y, coefs):
    # valores y previstos
    previsoes = [f(coefs, x) for x in dataset_x]
    
    # Calcula o erro absoluto médio
    erro_absoluto = tf.reduce_mean(tf.abs(tf.constant(dataset_y, dtype=tf.float32) - tf.constant(previsoes, dtype=tf.float32)))
    
    return erro_absoluto.numpy()

## Utilização
### Grau 1
$$
f(x) = a_1x+a_2
$$

In [58]:
grau = 1
coefs = grau + 1

# Target randomico
# coefs_target = [float(round(random.random(), 2)) for _ in range(coefs)]
coefs_target = [7,7]
a = tf.Variable(coefs_target, name='coefs', dtype=tf.float32)
print("coeficientes esperados:",coefs_target)

# dataset_x = gerar_sequencia(10,-2,2)
dataset_x = [0.0  ,1.0   ,2.0   ,3.0   ,4.0    ,5.0   ,6.0]
dataset_y = [f(a, x).numpy() for x in dataset_x]

gd, i = grad_desc(grau, dataset_x, dataset_y,10**(-4),0.01)
print(gd,i)

acuracia = calcular_acuracia(dataset_x, dataset_y, gd)
print("Erro Absoluto Médio:", acuracia)

coeficientes esperados: [7, 7]
[7.0005555 6.9976473] 185
Erro Absoluto Médio: 0.0010873249


Para valores e coeficientes randômicos.

In [53]:
grau = 1
coefs = grau + 1

coefs_target = [round(random.uniform(-10, 10), 2) for _ in range(coefs)]

a = tf.Variable(coefs_target, name='coefs', dtype=tf.float32)
print("coeficientes esperados:",coefs_target)

dataset_x = gerar_sequencia(30,-2,2)
dataset_y = [f(a, x).numpy() for x in dataset_x]

gd, i = grad_desc(grau, dataset_x, dataset_y,10**(-4),0.001)
print(gd,i)

acuracia = calcular_acuracia(dataset_x, dataset_y, gd)
print("Erro Absoluto Médio:", acuracia)

coeficientes esperados: [-9.53, -0.04]
[-9.53       -0.04010295] 11
Erro Absoluto Médio: 0.000102734564


### grau 3
$$
f(x) = a_1x^3+a_2x^2+a_3x+a_4
$$

In [65]:
grau = 2
coefs = grau + 1

# Target randomico
coefs_target = [round(random.uniform(-5, 5), 2) for _ in range(coefs)]

a = tf.Variable(coefs_target, name='coefs', dtype=tf.float32)
print("coeficientes esperados:",coefs_target)

dataset_x = gerar_sequencia(200,-20,20)
dataset_y = [f(a, x).numpy() for x in dataset_x]

gd, i = grad_desc(grau, dataset_x, dataset_y,10**(-16),0.01)
print(gd,i)


acuracia = calcular_acuracia(dataset_x, dataset_y, gd)
print("Erro Absoluto Médio:", acuracia)

coeficientes esperados: [1.04, 2.76, -1.71]
[ 6.687576e+35 -9.903459e+26  2.758893e+33] 8
Erro Absoluto Médio: inf


In [ ]:
##### grau = 17
coefs = grau + 1

# Target randomico
#coefs_target = [round(random.random(), 2) for _ in range(coefs)]
coefs_target = [random.uniform(-3, 3) for _ in range(coefs)]

print("coeficientes esperados:",coefs_target)

# Gerando data_set com base nos coefs_target
a = tf.Variable(coefs_target, name='coefs')

dataset_x = gerar_sequencia(10,-2,2)
dataset_y = [f(a, x).numpy() for x in dataset_x]

gd, i = grad_desc(grau, dataset_x, dataset_y,10**(-16),0.01)
print(gd,i)